In [1]:
import pandas as pd

In [2]:
file_path = '../data/code_terms.xlsx'
term_codes = pd.read_excel(file_path)

In [3]:
term_codes = term_codes.iloc[:,0:3]

In [4]:
term_codes.columns

Index(['code', 'level', 'term'], dtype='object')

In [5]:
term_codes.columns = ["code","level","term"]

In [6]:
term_codes.head(10)

,code,level,term
0,ICDO3.2,Level,Term
1,NaN,1,MORPHOLOGY
2,800,2,"Neoplasms, NOS"
3,8000/0,Preferred,"Neoplasm, benign"
4,8000/0,Synonym,"Tumor, benign"
5,8000/0,Synonym,"Unclassified tumor, benign"
6,8000/1,Preferred,"Neoplasm, uncertain whether benign or malignant"
7,8000/1,Synonym,"Neoplasm, NOS"
8,8000/1,Synonym,"Tumor, NOS"
9,8000/1,Synonym,"Unclassified tumor, borderline malignancy"


In [7]:
print("""Example Output:
    8000/0 Neoplasm, benign 800 Neoplasms, NOS
    8000/0 Tumor, benign    800 Neoplasms, NOS
    8000/1 Tumor, NOS       800 Neoplasms, NOS
    """)

Example Output:
    8000/0 Neoplasm, benign 800 Neoplasms, NOS
    8000/0 Tumor, benign    800 Neoplasms, NOS
    8000/1 Tumor, NOS       800 Neoplasms, NOS
    


In [8]:
term_groups = term_codes[term_codes["level"]==2]

In [9]:
term_groups.head(3)

,code,level,term
2,800,2,"Neoplasms, NOS"
34,801-804,2,"Epithelial neoplasms, NOS"
88,805-808,2,Squamous cell neoplasms


In [10]:
delta = term_groups['code'].str.split("-", expand = True)
delta.head(2)

,0,1
2,800,None
34,801,804


In [11]:
term_groups['range_start'] = delta[0]
term_groups['range_end'] = delta[1]

C:\Users\thangavelucs\AppData\Local\Temp\1\ipykernel_28480\2115728192.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  term_groups['range_start'] = delta[0]
C:\Users\thangavelucs\AppData\Local\Temp\1\ipykernel_28480\2115728192.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  term_groups['range_end'] = delta[1]


In [12]:
term_groups.head(5)

,code,level,term,range_start,range_end
2,800,2,"Neoplasms, NOS",800,None
34,801-804,2,"Epithelial neoplasms, NOS",801,804
88,805-808,2,Squamous cell neoplasms,805,808
185,809-811,2,Basal cell neoplasms,809,811
238,812-813,2,Transitional cell papillomas and carcinomas,812,813


In [13]:
df = pd.DataFrame()

for idx,row in term_groups.iterrows():
    if pd.isna(row['range_end']):
        new_row = row.copy()
        new_row['key'] = row['range_start']
        df = pd.concat([df, new_row.to_frame().T], ignore_index=True)
    else:
        start = int(row['range_start'])
        end = int(row['range_end'])
        for i in list(range(start, end + 1)):  # Wrap in list()
            new_row = row.copy()
            new_row['key'] = i
            df = pd.concat([df, new_row.to_frame().T], ignore_index=True)

In [14]:
#df.columns = [["prefix","level","group","range_start","range_end","key"]]
df.head(25)

,code,level,term,range_start,range_end,key
0,800,2,"Neoplasms, NOS",800,None,800
1,801-804,2,"Epithelial neoplasms, NOS",801,804,801
2,801-804,2,"Epithelial neoplasms, NOS",801,804,802
3,801-804,2,"Epithelial neoplasms, NOS",801,804,803
4,801-804,2,"Epithelial neoplasms, NOS",801,804,804
5,805-808,2,Squamous cell neoplasms,805,808,805
6,805-808,2,Squamous cell neoplasms,805,808,806
7,805-808,2,Squamous cell neoplasms,805,808,807
8,805-808,2,Squamous cell neoplasms,805,808,808
9,809-811,2,Basal cell neoplasms,809,811,809


In [15]:
# Step 1: Load the original diagnosis data (the one with 4-digit codes)
# This should be your df with Preferred/Synonym rows
diagnosis_df = term_codes[term_codes['level'].isin(['Preferred', 'Synonym'])].copy()

# Step 2: Extract first 3 digits from the 4-digit code
diagnosis_df['key'] = diagnosis_df['code'].str[:3]

# Step 3: Merge with your glossary (the expanded df you just made)
# Rename your expanded glossary columns if needed
glossary = df  # Your expanded dataframe with the 'key' column


# Step 4: Merge/join on the 'key' column
diagnosis_df['key'] = diagnosis_df['key'].astype(str).str.strip()
glossary['key'] = glossary['key'].astype(str).str.strip()
mapping_output = diagnosis_df.merge(
    glossary[['key', 'code', 'term']], 
    on='key', 
    how='left'
)

In [16]:
mapping_output

,code_x,level,term_x,key,code_y,term_y
0,8000/0,Preferred,"Neoplasm, benign",800,800,"Neoplasms, NOS"
1,8000/0,Synonym,"Tumor, benign",800,800,"Neoplasms, NOS"
2,8000/0,Synonym,"Unclassified tumor, benign",800,800,"Neoplasms, NOS"
3,8000/1,Preferred,"Neoplasm, uncertain whether benign or malignant",800,800,"Neoplasms, NOS"
4,8000/1,Synonym,"Neoplasm, NOS",800,800,"Neoplasms, NOS"
...,...,...,...,...,...,...
2326,9987/3,Synonym,"Therapy-related myelodysplastic syndrome, epip...",998,998-999,Myelodysplastic syndromes
2327,9989/3,Preferred,"Myelodysplastic syndrome, NOS",998,998-999,Myelodysplastic syndromes
2328,9989/3,Synonym,Preleukemia,998,998-999,Myelodysplastic syndromes
2329,9989/3,Synonym,Preleukemic syndrome,998,998-999,Myelodysplastic syndromes


In [17]:
mapping_output.to_csv('../data/draft_mapping_file.tsv', sep='\t', index=False)

In [18]:
mapping_output.to_csv('../data/draft_mapping_file.csv', sep='\t', index=False)

In [19]:
# Check the glossary structure
print("Glossary columns:", glossary.columns.tolist())
print("\nFirst few rows of glossary:")
print(glossary.head(10))
print("\nGlossary 'key' column data type:", glossary['key'].dtype)

Glossary columns: ['code', 'level', 'term', 'range_start', 'range_end', 'key']

First few rows of glossary:
      code level                       term range_start range_end  key
0      800     2             Neoplasms, NOS         800      None  800
1  801-804     2  Epithelial neoplasms, NOS         801       804  801
2  801-804     2  Epithelial neoplasms, NOS         801       804  802
3  801-804     2  Epithelial neoplasms, NOS         801       804  803
4  801-804     2  Epithelial neoplasms, NOS         801       804  804
5  805-808     2    Squamous cell neoplasms         805       808  805
6  805-808     2    Squamous cell neoplasms         805       808  806
7  805-808     2    Squamous cell neoplasms         805       808  807
8  805-808     2    Squamous cell neoplasms         805       808  808
9  809-811     2       Basal cell neoplasms         809       811  809

Glossary 'key' column data type: object


In [20]:
# Check data types
print("diagnosis_df key type:", diagnosis_df['key'].dtype)
print("glossary key type:", glossary['key'].dtype)

diagnosis_df key type: object
glossary key type: object


In [21]:
# Step 5: Select and rename columns for final output
# diagnosis_code_4digit | diagnosis_term_4digit | group_code_3digit | group_term_3digit
final_mapping = mapping_output[['ICD03.2', 'Term', 'code', 'term']].copy()
final_mapping.columns = ['diagnosis_code_4digit', 'diagnosis_term_4digit', 'group_code_3digit', 'group_term_3digit']

# Step 6: Check it
print(final_mapping.head(20))

# Step 7: Save to TSV
final_mapping.to_csv('../output/icdo_4digit_to_3digit_mapping.tsv', sep='\t', index=False)

KeyError: "None of [Index(['ICD03.2', 'Term', 'code', 'term'], dtype='object')] are in the [columns]"

### ScratchPad

In [ ]:
#Chloe's Draft - try to make similar to above
'''
for row in term_groups.interrows():
    if end.isna():
        code = row[0]
        new_row = row.append(code)
        df.append(new_row)
    else:
        start = row[3]
        end = row[4]
        for i in range(start, end+1):
            code = i
            new_row = row.append(code)
            df.append(new_row)
'''